# Road Accident Risk Analysis — India (v2, statistically validated)
Loads dataset, checks distribution uniformity, runs chi-square tests for Hour/Weather/State/Road Type vs Severity, and computes population-adjusted state rates.

In [ ]:
df <- read.csv("accident_prediction_india.csv")
head(df)

## 0. Dataset reliability check
Check how uniform each categorical column is. Near-equal category counts across the board suggest randomly generated data.

In [ ]:
cols_to_check <- c("Weather.Conditions","Road.Type","Day.of.Week","Month","Accident.Severity","Road.Condition","Lighting.Conditions")
for (col in cols_to_check) {
  tab <- table(df[[col]])
  cat(col, "-> min:", min(tab), "max:", max(tab), "ratio:", round(max(tab)/min(tab),2), "\n")
}

In [ ]:
records_per_state <- table(df$State.Name)
cat("Records per state -> min:", min(records_per_state), "max:", max(records_per_state), "\n")
cat("(Compare to real population range: ~7 lakh to ~24 crore -- a 340x difference not reflected here)\n")

## 1. Hour vs Severity

In [ ]:
df$Hour <- as.integer(sapply(strsplit(as.character(df$Time.of.Day), ":"), `[`, 1))
hour_table <- table(df$Hour, df$Accident.Severity)
chisq.test(hour_table)

## 2. Weather vs Severity

In [ ]:
weather_table <- table(df$Weather.Conditions, df$Accident.Severity)
print(weather_table)
chisq.test(weather_table)

## 3. Road Type vs Severity

In [ ]:
road_table <- table(df$Road.Type, df$Accident.Severity)
print(road_table)
chisq.test(road_table)

## 4. State vs Severity

In [ ]:
state_table <- table(df$State.Name, df$Accident.Severity)
chisq.test(state_table)

## 5. Population-adjusted state fatality rates
State population figures below are 2025 estimates (Govt. of India Technical Group Population Projection Report 2011-2036, via StatisticsTimes.com).

In [ ]:
state_population <- c(
  "Jammu and Kashmir"=13831000, "Uttar Pradesh"=241265000, "Chhattisgarh"=30982000,
  "Sikkim"=703000, "Meghalaya"=3417000, "Himachal Pradesh"=7555000, "Rajasthan"=83060000,
  "Assam"=36493000, "Bihar"=131041000, "Telangana"=38499000, "Arunachal Pradesh"=1594000,
  "Andhra Pradesh"=53586000, "Karnataka"=68679000, "Madhya Pradesh"=88985000, "Puducherry"=1732000,
  "Maharashtra"=128659000, "Tamil Nadu"=77394000, "Chandigarh"=1259000, "Gujarat"=73513000,
  "Odisha"=46953000, "West Bengal"=100202000, "Kerala"=36111000, "Nagaland"=2279000,
  "Tripura"=4232000, "Uttarakhand"=11913000, "Haryana"=31057000, "Goa"=1593000,
  "Mizoram"=1264000, "Delhi"=22277000, "Jharkhand"=40626000, "Punjab"=31188000, "Manipur"=3289000
)

fatal_df <- df[df$Accident.Severity == "Fatal", ]
state_fatal <- as.data.frame(table(fatal_df$State.Name))
colnames(state_fatal) <- c("State", "Fatal_Count")
state_fatal$Population <- state_population[as.character(state_fatal$State)]
state_fatal$Fatal_per_Crore <- round(state_fatal$Fatal_Count / (state_fatal$Population/1e7), 2)

cat("Top 10 by RAW COUNT:\n")
print(head(state_fatal[order(-state_fatal$Fatal_Count), c("State","Fatal_Count")], 10))

cat("\nTop 10 by RATE PER CRORE POPULATION:\n")
print(head(state_fatal[order(-state_fatal$Fatal_per_Crore), c("State","Fatal_Count","Fatal_per_Crore")], 10))

## Summary
All four chi-square tests (Hour, Weather, State, Road Type vs Severity) return p > 0.05 -- none statistically significant. Combined with the near-uniform distribution of every categorical column (Section 0), the dataset behaves like randomly generated practice data rather than real accident records. State-level raw counts and population-adjusted rates produce completely different rankings, demonstrating why normalization matters before drawing conclusions.